![Piksel Sandbox](../../assets/piksel_header.png)

# Plotting

## A. Visualising xarray data

xarray DataArrays expose a `.plot` accessor that wraps matplotlib.
The accessor inspects the DataArray's dimensions and chooses an appropriate figure: a two-dimensional array becomes a colour-mapped image, three bands stacked along a new dimension become an RGB composite, and an extra `time` (or other) dimension can split the figure into a row of panels.

Three plot calls in this notebook cover the most common visualisations of an Earth observation dataset.
Each is built against the same Lake Toba dataset used in notebook 04.

## B. Outline

1. Load the same Lake Toba dataset used in notebook 04.
2. Plot a single band as a colour-mapped image.
3. Stack red, green, and blue into a true-colour composite.
4. Split a band across time into a row of facet panels.

## C. Loading a sample dataset

The load call reuses the area, grid, and product from notebooks 03 and 04 over a two-year range.
matplotlib is imported alongside datacube so every plot in the notebook draws into an inline figure.

In [ ]:
import datacube
import matplotlib.pyplot as plt

dc = datacube.Datacube(app="plotting")

query = {
    "product": "s2_geomad_annual",
    "x": (98.80, 98.90),
    "y": (2.65, 2.55),
    "time": ("2024", "2025"),
    "measurements": ["red", "green", "blue", "nir"],
    "output_crs": "EPSG:32647",
    "resolution": (-30, 30),
}

ds = dc.load(**query)

## D. Single-band image

A two-dimensional DataArray plots as a colour-mapped image with axis labels taken from its coordinates.
Choosing the band and the time slice uses `.sel` or `.isel` from notebook 04.
The `cmap` argument selects the colour scheme; `"gray"` is the conventional choice for inspecting a single band, mapping low reflectance to black and high reflectance to white.

In [ ]:
ds.red.isel(time=0).plot(cmap="gray")

The red band from the first time slice draws as one image.
The axes carry the projected metres of the `y` and `x` coordinates, and the colourbar carries the reflectance values of the variable.
Bright water-edge sediment and bare ground appear light; deep water appears near black; vegetation sits in the middle of the scale.

## E. True-colour composite

A true-colour composite stacks the red, green, and blue bands into a single figure that approximates what the human eye sees.
xarray exposes this through `.plot.imshow(rgb=...)`: the named dimension supplies the three colour channels.
The bands must first be assembled into one DataArray along a new dimension, using `.to_array`.

In [ ]:
rgb = ds[["red", "green", "blue"]].to_array(dim="band").isel(time=0)

rgb.plot.imshow(rgb="band", vmin=0, vmax=3000)

The composite shows Lake Toba's open water as dark, surrounding vegetation as green, and bare or built-up patches as brighter neutrals.

The `vmin` and `vmax` arguments set the lower and upper ends of the reflectance range used for the display stretch.
Surface reflectance values from `dc.load` are scaled integers, with most land-surface pixels falling between roughly 0 and 3000; clipping the stretch at 3000 keeps the image from being washed out by occasional bright pixels.

The result is a 2D image with the same `y` and `x` axes as a single-band plot, with three colour channels in place of a colourmap.

## F. Faceted time steps

A DataArray with an extra dimension beyond `y` and `x` can be drawn as a row or grid of panels, one panel per coordinate value along that extra dimension.
`.plot(col="time")` produces a row of panels with one image per time step.
This is the natural shape for comparing the same area across years.

In [ ]:
ds.red.plot(col="time", cmap="Reds", vmin=0, vmax=3000)

The two panels show the red band for 2024 and 2025 side by side.
The colour scale is shared across both panels, so brightness differences between years are comparable directly.

Any extra coordinate dimension can drive a facet layout the same way.
`col=` and `row=` select the dimensions to split on, and `col_wrap=` controls when a long row of panels breaks onto a new row.

## G. Next steps

Notebook 06 uses these plotting patterns to display spectral indices like NDVI and to compare vegetation across several years.
Continue to [`06_basic_analysis.ipynb`](./06_basic_analysis.ipynb).